In [0]:
dbutils.library.restartPython()

In [0]:
import sys, os

here = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    .notebookPath().get()
)
probe = "/Workspace" + here
SRC = None
for _ in range(6):
    candidate = os.path.join(probe, "src")
    if os.path.isdir(os.path.join(candidate, "agentic_ai")):
        SRC = candidate
        break
    probe = os.path.dirname(probe)

assert SRC, "could not locate src/agentic_ai above this notebook"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("SRC:", SRC)

In [0]:
tel = os.path.join(SRC, "agentic_ai", "telemetry")
rem = os.path.join(SRC, "agentic_ai", "remediation")
print("telemetry/:", os.listdir(tel))
print("remediation/:", os.listdir(rem))

In [0]:
from agentic_ai.config import Settings

s = Settings(catalog="databricks_ws", schema="agentic_ai_dev")
print("detection_source:", s.detection_source)
print("failed_states:", s.failed_states)
print("approval_expiry_hours:", s.approval_expiry_hours)
print("fq_schema:", s.fq_schema)

In [0]:
from agentic_ai.telemetry.sources import get_source, default_since

src = get_source(s)
print("source:", src.name)
print("failed runs visible:", len(src.fetch_failed_runs(default_since(s), s.watcher_batch_limit)))

In [0]:
from agentic_ai.migrations import apply_all

print(apply_all(s))
print(apply_all(s))
display(spark.sql(f"SHOW TABLES IN {s.fq_schema}"))

In [0]:
from agentic_ai.telemetry.watcher import run_once as watcher_run_once, get_watermark

print("watermark:", get_watermark(s))
count = watcher_run_once(s)
print("incidents raised this pass:", count)

In [0]:
display(spark.sql(f"SELECT status, count(*) AS n FROM {s.table('incidents')} GROUP BY status"))
display(spark.sql(f"SELECT * FROM {s.table('incidents')} ORDER BY created_at DESC LIMIT 10"))
display(spark.sql(f"SELECT * FROM {s.table('watcher_state')}"))

In [0]:
from agentic_ai.remediation.reconciler import run_once as reconciler_run_once
result = reconciler_run_once(s)
print(result)

In [0]:
display(spark.sql(f"SELECT incident_id, status, classified_agent, severity FROM {s.table('incidents')} ORDER BY updated_at DESC LIMIT 10"))
display(spark.sql(f"SELECT * FROM {s.table('approval_requests')} ORDER BY requested_at DESC LIMIT 10"))
display(spark.sql(f"SELECT * FROM {s.table('remediation_log')} ORDER BY executed_at DESC LIMIT 10"))

In [0]:
pending = spark.sql(f"SELECT request_id FROM {s.table('approval_requests')} WHERE status = 'pending' LIMIT 1").collect()
if pending:
    rid = pending[0]["request_id"]
    spark.sql(f"UPDATE {s.table('approval_requests')} SET status = 'approved' WHERE request_id = '{rid}'")
    print("approved:", rid)
else:
    print("no pending requests to approve")

In [0]:
result2 = reconciler_run_once(s)
print(result2)
display(spark.sql(f"SELECT * FROM {s.table('remediation_log')} ORDER BY executed_at DESC LIMIT 10"))

In [0]:
result3 = reconciler_run_once(s)
print(result3)

In [0]:
display(spark.sql(f"""
    SELECT incident_id, action_type, execution_status, action_text, error_message
    FROM {s.table('remediation_log')}
    ORDER BY executed_at DESC
"""))